<a href="https://colab.research.google.com/github/hfakhimi5058-tech/OC-MSI-Oral-Cancer-AI/blob/main/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# ORAL CANCER OC-MSI
# ConvNeXt V2 vs DINOv2
# Google Colab - Complete Pipeline
# ============================================================

# -----------------------------
# 1. INSTALL LIBRARIES
# -----------------------------

!pip -q install timm scikit-learn seaborn pandas matplotlib kagglehub

# -----------------------------
# 2. IMPORTS
# -----------------------------

import os
import glob
import copy
import random
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    auc
)

from tqdm.auto import tqdm


# -----------------------------
# 3. RANDOM SEED
# -----------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# -----------------------------
# 4. DEVICE
# -----------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("DEVICE:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled.")
    print("Go to Runtime > Change runtime type > GPU")

print("=" * 60)


# ============================================================
# 5. DOWNLOAD DATASET FROM KAGGLE
# ============================================================

print("\nDownloading OC-MSI dataset...")

import kagglehub

DATASET_PATH = kagglehub.dataset_download(
    "zoya77/oral-cancer-multispectral-imaging-dataset-oc-msi"
)

print("\nDataset downloaded to

SyntaxError: unterminated string literal (detected at line 100) (2357676801.py, line 100)

In [ ]:
# ============================================================
# STEP 18 — INSTALL MODERN VISION MODEL DEPENDENCIES
# SwinV2-Tiny
# ============================================================

!pip -q install timm

In [ ]:
# ============================================================
# STEP 19 — VERIFY GPU + SWINV2-TINY
# ============================================================

import torch
import timm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

# Find available SwinV2 models
swin_models = timm.list_models("*swinv2*")

print("\nAvailable SwinV2 models:")
for name in swin_models[:30]:
    print(" -", name)

print("\nTotal SwinV2 models found:", len(swin_models))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

Available SwinV2 models:
 - swinv2_base_window8_256
 - swinv2_base_window12_192
 - swinv2_base_window12to16_192to256
 - swinv2_base_window12to24_192to384
 - swinv2_base_window16_256
 - swinv2_cr_base_224
 - swinv2_cr_base_384
 - swinv2_cr_base_ns_224
 - swinv2_cr_giant_224
 - swinv2_cr_giant_384
 - swinv2_cr_huge_224
 - swinv2_cr_huge_384
 - swinv2_cr_large_224
 - swinv2_cr_large_384
 - swinv2_cr_small_224
 - swinv2_cr_small_384
 - swinv2_cr_small_ns_224
 - swinv2_cr_small_ns_256
 - swinv2_cr_tiny_224
 - swinv2_cr_tiny_384
 - swinv2_cr_tiny_ns_224
 - swinv2_large_window12_192
 - swinv2_large_window12to16_192to256
 - swinv2_large_window12to24_192to384
 - swinv2_small_window8_256
 - swinv2_small_window16_256
 - swinv2_tiny_window8_256
 - swinv2_tiny_window16_256

Total SwinV2 models found: 28


In [ ]:
# ============================================================
# STEP 20 — LOAD SWINV2-TINY
# ============================================================

import torch
import timm

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# Create SwinV2-Tiny
# ------------------------------------------------------------

swin_model = timm.create_model(
    "swinv2_tiny_window8_256",
    pretrained=True,
    num_classes=2
)

swin_model = swin_model.to(device)


# ------------------------------------------------------------
# Model information
# ------------------------------------------------------------

total_params = sum(p.numel() for p in swin_model.parameters())
trainable_params = sum(
    p.numel() for p in swin_model.parameters()
    if p.requires_grad
)

print("\nModel:", "SwinV2-Tiny")
print("Input size: 256 × 256")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\nSwinV2-Tiny loaded successfully.")

Device: cuda
GPU: Tesla T4


model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            


Model: SwinV2-Tiny
Input size: 256 × 256
Total parameters: 27,579,692
Trainable parameters: 27,579,692

SwinV2-Tiny loaded successfully.


In [ ]:
# ============================================================
# STEP 21 — SWINV2-TINY DATA LOADERS
# ============================================================

import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

DATA_ROOT = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split"

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

print("Train:", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Test:", TEST_DIR)


# ------------------------------------------------------------
# ImageNet normalization
# SwinV2 pretrained weights expect ImageNet-style normalization
# ------------------------------------------------------------

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ------------------------------------------------------------
# Training augmentation
# Keep augmentation comparable to EfficientNet experiment
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.92, 1.08)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.05
    ),

    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


# ------------------------------------------------------------
# Validation / Test
# NO augmentation
# ------------------------------------------------------------

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

BATCH_SIZE_SWIN = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_SWIN,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_SWIN,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_SWIN,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Verify class mapping and sample counts
# ------------------------------------------------------------

print("\nClass mapping:")
print(train_dataset.class_to_idx)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nNumber of batches:")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

Train: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/train
Validation: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/val
Test: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/test


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/train'

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder created:")
print(PROJECT_DIR)

Project folder created:
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI


In [ ]:
import os
import glob

print("Checking current runtime files...\n")

for path in [
    "/kaggle/working",
    "/content",
    "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"
]:
    print(f"\n{path}")
    if os.path.exists(path):
        files = glob.glob(os.path.join(path, "*"))
        print(files[:20])
    else:
        print("Not found")

Checking current runtime files...


/kaggle/working
['/kaggle/working/best_OC_MSI_EfficientNetB0_v2.keras', '/kaggle/working/best_OC_MSI_EfficientNetB0_finetuned.keras', '/kaggle/working/best_OC_MSI_EfficientNetB0.keras', '/kaggle/working/OC_MSI_Split']

/content
['/content/drive', '/content/sample_data']

/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI
[]


In [ ]:
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"

# Create folders
models_dir = os.path.join(PROJECT_DIR, "models")
split_dir = os.path.join(PROJECT_DIR, "OC_MSI_Split")

os.makedirs(models_dir, exist_ok=True)

# Copy trained models
model_files = [
    "/kaggle/working/best_OC_MSI_EfficientNetB0_v2.keras",
    "/kaggle/working/best_OC_MSI_EfficientNetB0_finetuned.keras",
    "/kaggle/working/best_OC_MSI_EfficientNetB0.keras"
]

for src in model_files:
    if os.path.exists(src):
        dst = os.path.join(models_dir, os.path.basename(src))
        shutil.copy2(src, dst)
        print("Saved:", dst)
    else:
        print("NOT FOUND:", src)

# Copy leakage-safe dataset split
src_split = "/kaggle/working/OC_MSI_Split"

if os.path.exists(src_split):
    if os.path.exists(split_dir):
        shutil.rmtree(split_dir)

    shutil.copytree(src_split, split_dir)
    print("\nDataset split saved:", split_dir)
else:
    print("Split folder not found.")

print("\nDONE — project backup completed.")

Saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/models/best_OC_MSI_EfficientNetB0_v2.keras
Saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/models/best_OC_MSI_EfficientNetB0_finetuned.keras
Saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/models/best_OC_MSI_EfficientNetB0.keras

Dataset split saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split

DONE — project backup completed.


In [ ]:
import os
import json
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Final locked threshold selected on validation set
FINAL_THRESHOLD = 0.4558

# Final test metrics
results = {
    "model": "EfficientNetB0",
    "task": "Oral Cancer Classification",
    "dataset": "OC-MSI",
    "input_size": "224x224x3",
    "validation_threshold": FINAL_THRESHOLD,
    "test_accuracy": 0.8767,
    "test_sensitivity_cancer": 0.8228,
    "test_specificity": 0.9403,
    "test_precision_cancer": 0.9420,
    "test_f1_cancer": 0.8784,
    "test_macro_f1": 0.8767,
    "validation_roc_auc": 0.9754,
    "validation_pr_auc": 0.9774,
    "validation_sensitivity": 0.9067,
    "validation_specificity": 0.9577,
    "test_samples": 146,
    "test_cancer": 79,
    "test_non_cancer": 67
}

# Save JSON
json_path = os.path.join(RESULTS_DIR, "final_test_results.json")

with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

# Save CSV
csv_path = os.path.join(RESULTS_DIR, "final_test_results.csv")

pd.DataFrame([results]).to_csv(csv_path, index=False)

print("Results saved successfully:")
print(json_path)
print(csv_path)

Results saved successfully:
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/final_test_results.json
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/final_test_results.csv


In [ ]:

import os

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"

readme = """# OC-MSI Oral Cancer AI

## AI-Assisted Oral Cancer Classification Using EfficientNetB0

This repository contains a research-oriented deep learning pipeline for
binary classification of oral cancer and non-cancer images using the
OC-MSI dataset and EfficientNetB0.

### Researcher

**Haleh Fakhimi Rezaei**

Dentistry | Genetics | Biotechnology | Artificial Intelligence in Oral Cancer Research

### Model

- Architecture: EfficientNetB0
- Transfer learning: ImageNet pretrained weights
- Input size: 224 × 224 × 3
- Data augmentation: rotation, translation, zoom, horizontal flip, brightness variation
- Fine-tuning: top layers of EfficientNetB0
- Binary classification: Cancer vs Non-Cancer

### Dataset

OC-MSI (Oral Cancer Multispectral Imaging Dataset)

Dataset composition:

- Total images: 950
- Cancer: 500
- Non-Cancer: 450

Exact duplicate images were identified and grouped to reduce the risk of
data leakage between train, validation, and test sets.

### Data Split

- Training: 658 images
- Validation: 146 images
- Test: 146 images

### Validation Performance

- ROC-AUC: 0.9754
- PR-AUC: 0.9774
- Sensitivity: 90.67%
- Specificity: 95.77%

The classification threshold was selected using the validation set only.

### Held-Out Test Performance

- Accuracy: 87.67%
- Cancer sensitivity: 82.28%
- Specificity: 94.03%
- Cancer precision: 94.20%
- Cancer F1-score: 87.84%

The test set was evaluated using a fixed threshold selected from the
validation set.

### Important Scientific Notes

This project is a research prototype for image classification and is
not a clinical diagnostic system.

Exact duplicate grouping was performed, but patient-level independence
could not be confirmed from the available dataset metadata. Therefore,
patient-level generalization should not be assumed.

The dataset is referred to as multispectral by its source; the image
files used in this project were processed as RGB images.

### Reproducibility

The repository is intended to document the complete experimental workflow,
including dataset inspection, leakage-aware splitting, model training,
fine-tuning, validation-based threshold selection, and held-out test
evaluation.

### Author

Haleh Fakhimi Rezaei
"""

readme_path = os.path.join(PROJECT_DIR, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

print("README created:")
print(readme_path)

README created:
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/README.md


In [ ]:
# ============================================================
# STEP 25 — SWINV2-TINY FINE-TUNING
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import timm

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Load BEST Stage-1 model
# ------------------------------------------------------------

swin_model = timm.create_model(
    "swinv2_tiny_window8_256",
    pretrained=False,
    num_classes=2
)

swin_model.load_state_dict(
    torch.load(
        "/content/best_OC_MSI_SwinV2_Tiny_stage1.pth",
        map_location=device
    )
)

swin_model = swin_model.to(device)


# ------------------------------------------------------------
# 2. Freeze everything first
# ------------------------------------------------------------

for param in swin_model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# 3. Unfreeze last part of backbone
# ------------------------------------------------------------

children = list(swin_model.children())

print("Model children:")

for i, child in enumerate(children):
    print(i, type(child).__name__)


# ------------------------------------------------------------
# 4. Unfreeze final layers
# ------------------------------------------------------------

for param in swin_model.layers[-1].parameters():
    param.requires_grad = True

for param in swin_model.norm.parameters():
    param.requires_grad = True

for param in swin_model.head.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 5. Loss
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 6. Optimizer
# ------------------------------------------------------------

optimizer = optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        swin_model.parameters()
    ),
    lr=1e-5,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# 7. Count trainable parameters
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in swin_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in swin_model.parameters()
    if p.requires_grad
)

print("\nFine-tuning setup:")
print("Total parameters:", f"{total_params:,}")
print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)
print(
    "Frozen parameters:",
    f"{total_params - trainable_params:,}"
)

print("Learning rate:", 1e-5)

NameError: name 'device' is not defined